# 01 Dataset Overview

**Scope.** Summarize the benchmark dataset structure, inspect the processed molecule table and label resources, and provide non-destructive integrity checks without fabricating absent data.

## Expected input files

- `data/processed/benchmark_molecules_4495.csv`
- `data/processed/odor_label_vocabulary_112.csv`
- `data/processed/odor_label_matrix_4495x112.csv`
- `data/interim/label_harmonization_table.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

SAVE_OUTPUTS = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_configs = (candidate / "configs").is_dir()
        has_readme = (candidate / "README.md").exists()
        if has_configs and has_readme:
            return candidate
        if has_configs:
            fallback = candidate
            return fallback
    raise FileNotFoundError("Could not locate project root containing configs/.")


def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def check_files(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({"path": path.as_posix(), "exists": path.exists()})
    return pd.DataFrame(rows)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        formatted = "
".join(f"- {path.as_posix()}" for path in missing)
        raise FileNotFoundError(
            "The following required files are missing:
"
            f"{formatted}
"
            "Data required from user later before this notebook can run end-to-end."
        )


def print_missing_file_report(report_df: pd.DataFrame) -> None:
    missing = report_df.loc[~report_df["exists"], "path"].tolist()
    if missing:
        print("Data required from user later:")
        for item in missing:
            print(f"- {item}")
    else:
        print("All expected input files are present.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "configs"
DATASET_CFG = load_yaml(CONFIG_DIR / "dataset.yaml")


## 1. Scope

This notebook summarizes benchmark dataset structure and performs lightweight checks on molecule-level and label-level processed tables. It does not create benchmark outputs unless `SAVE_OUTPUTS = True` is set manually.

## 2. Expected inputs

The notebook expects the processed benchmark molecule table, standardized odor vocabulary, multi-hot label matrix, and the label harmonization table.

## 3. Load configuration

In [ ]:
expected_files = [
    PROJECT_ROOT / DATASET_CFG['files']['benchmark_molecules'],
    PROJECT_ROOT / DATASET_CFG['files']['odor_label_vocabulary'],
    PROJECT_ROOT / DATASET_CFG['files']['odor_label_matrix'],
    PROJECT_ROOT / DATASET_CFG['files']['label_harmonization_table'],
]
expected_counts = DATASET_CFG['dataset']
print('Project root:', PROJECT_ROOT.as_posix())
print('Expected benchmark molecules:', expected_counts['expected_n_molecules'])
print('Expected labels:', expected_counts['expected_n_labels'])

## 4. File existence checks

In [ ]:
file_report = check_files(expected_files)
display(file_report)
print_missing_file_report(file_report)

## 5. Load benchmark tables

In [ ]:
if file_report['exists'].all():
    benchmark = pd.read_csv(expected_files[0])
    vocabulary = pd.read_csv(expected_files[1])
    label_matrix = pd.read_csv(expected_files[2])
    harmonization = pd.read_csv(expected_files[3])
    print('Loaded benchmark tables successfully.')
else:
    print('Skipping data load until required files are available.')

## 6. Basic integrity checks

Checks include molecule count, label count, alignment between molecule IDs and the label matrix, and alignment between vocabulary entries and label-matrix columns.

In [ ]:
if file_report['exists'].all():
    molecule_count_ok = len(benchmark) == expected_counts['expected_n_molecules']
    label_columns = [c for c in label_matrix.columns if c != 'molecule_id']
    label_count_ok = len(label_columns) == expected_counts['expected_n_labels']
    molecule_ids_align = benchmark['molecule_id'].astype(str).tolist() == label_matrix['molecule_id'].astype(str).tolist()
    vocab_labels = vocabulary['standard_label'].astype(str).tolist() if 'standard_label' in vocabulary.columns else []
    label_columns_align = label_columns == vocab_labels if vocab_labels else False
    integrity_summary = pd.DataFrame([
        {'check': 'expected molecule count', 'passed': molecule_count_ok},
        {'check': 'expected label count', 'passed': label_count_ok},
        {'check': 'molecule IDs align', 'passed': molecule_ids_align},
        {'check': 'label columns align with vocabulary', 'passed': label_columns_align},
    ])
    display(integrity_summary)
else:
    print('Data required from user later before integrity checks can run.')

## 7. Label frequency summary

In [ ]:
if file_report['exists'].all():
    label_columns = [c for c in label_matrix.columns if c != 'molecule_id']
    label_frequency = label_matrix[label_columns].sum(axis=0).sort_values(ascending=False).rename('frequency')
    display(label_frequency.head(20).to_frame())
    ax = label_frequency.head(20).sort_values().plot(kind='barh', figsize=(8, 6), title='Top 20 label frequencies')
    ax.set_xlabel('Count')
    plt.tight_layout()
else:
    print('Label frequency summary requires the label matrix and vocabulary files.')

## 8. Label co-occurrence matrix preview

In [ ]:
if file_report['exists'].all():
    label_columns = [c for c in label_matrix.columns if c != 'molecule_id']
    top_labels = label_matrix[label_columns].sum(axis=0).sort_values(ascending=False).head(10).index.tolist()
    co_occurrence = label_matrix[top_labels].T.dot(label_matrix[top_labels])
    display(co_occurrence)
    plt.figure(figsize=(7, 6))
    plt.imshow(co_occurrence.values, cmap='viridis')
    plt.xticks(range(len(top_labels)), top_labels, rotation=90)
    plt.yticks(range(len(top_labels)), top_labels)
    plt.title('Top-label co-occurrence preview')
    plt.colorbar(label='Co-occurrence count')
    plt.tight_layout()
else:
    print('Co-occurrence preview skipped because required files are missing.')

## 9. Missing data checklist

In [ ]:
missing_items = file_report.loc[~file_report['exists'], 'path'].tolist()
if missing_items:
    print('Data required from user later:')
    for item in missing_items:
        print(f'- {item}')
else:
    print('No missing benchmark input files detected for this notebook.')